# A Dyalog workflow. Or two.

> I don't know how many of you have ever met Dijkstra, but you probably know that arrogance in computer science is measured in nano-Dijkstras.
\--_Alan Kay_ [@kay1997oopsla]

For the majority of the examples we'll encounter in this book, you can just type them directly into the Dyalog IDE, `Ride`, or even the [TryAPL](https://tryapl.org/) website, and when you understand the point made, scrap them. But as you progress, sooner or later you will want to save your work to disk. 

If you have a similar background to me, you'll expect to be able to write your code using an editor of your choice, arranging it into files containing logically related routines, which you then put into a source code revision management system, like git, perhaps triggering a build and test pipeline on every commit via something like [Jenkins](https://www.jenkins.io/) or [circle-ci](https://circleci.com/). This is most likely in the neighbourhood of your current workflow, regardless of which language or platform you're using, from [Ada](https://en.wikipedia.org/wiki/Ada_(programming_language)) to [Zig](https://ziglang.org/).

Before we get into the practicalities of how you can do this and more, we'll need a slight diversion into the history of APL and Dyalog in particular [[Kromb&Hui2020](https://dl.acm.org/doi/pdf/10.1145/3386319)]. Firstly, APL predates all these workflows and build tools. That in itself isn't an issue: so does C. But APL wasn't really conceived as a _software engineering_ tool. During its heyday, middle management and CFOs were slinging APL to generate forecasts and reports, and they _loved_ APL as it allowed them to essentially perform tasks that up until then had been in the realm of specialist "batch processing" on the company mainframe. It essentially occupied the niche which is now dominated by Microsoft Excel.

This influenced all APL vendors, including Dyalog. Commercial APL systems are fully integrated application stacks that make it seductively convenient to package up your work as binary workspaces that can be easily distributed to other APL users without ever having to "leave" the confines of the stack itself. Dyalog's tooling and development philosophy supports this world view, as that is what many of its customers expect: professional Dyalog developers working for billion-dollar financial institutions will be of the opinion that you can pry their binary workspaces from their cold, dead hands. You can build fully-fledged, GUI-driven applications integrating deeply with code bases written in .NET.

Yet, for you, the newcomer, this approach will probably feel a bit alien. Dyalog obviously understand this, and they're walking the tightrope of both delivering what their current paying, long-term customers need, but also a keen eye on what _new_ users and potential customers might be looking to. Morten Kromberg, Dyalog CTO, wrote a [blog post](https://www.dyalog.com/blog/2021/02/the-apl-orchard/) recently, ostensibly about the [APL Orchard](https://apl.chat) chat room, but also touching on some of Dyalog's challenges when dealing with this dichotomy.

> There are a couple of other topics that I should possibly also explain my views on, such as the dilemma of whether APL is a Notation or a Programming Language, and why Dyalog APL is not open source, but this post is very long already. Depending on the response I get to this post, I may return to that in the future. --_Morten Kromberg_

Fortunately, and for the avoidance of any doubt, you can _absolutely_ work with modern Dyalog (I'm using version 21.0 at the time of writing this) in a pretty similar way to what you expect, but it requires a little bit of setup to get right, and is not quite there "out of the box" at the time of writing. However, at the time of _reading_ some of the kinks may well have been ironed out: better support for Dyalog as a scripting engine, and working with code-as-text is a priority for Dyalog right now. 

We mentioned workspaces when we talked about namespaces earlier. Let's return to take another look at them and then at the `]LINK` tool that allows you to keep your code in text files that can be versioned.

:::{note} Other resources
:class: resources

Dyalog docs
: [Link](https://dyalog.github.io/link)

Webinar
: [Source Code Management with GitHub and APL](https://dyalog.tv/Webinar/?v=S3JBHDayVjU)

Video
: [Link v3.0](https://www.youtube.com/watch?v=K_-E1tnH06k)
:::

## Saving and loading workspaces

If you open `Ride`, you'll see no "Save as.." in the menu, and your OS's default key binding for "Save" does nothing. You might have figured out how to enter multi-line functions, and how to "fix" (somewhere between "save" and "compile") them. Yet, when you "fix" a function, where does it go? 

At no point were you asked for a file name. Again, the Excel analogy might help to clarify. Think of a cell containing a formula in Excel -- you're not expecting to "save" the formula out to a separate file. Instead, the formula becomes an integral part of the sheet or workbook. We can think of Dyalog workspaces as analogous to Excel workbooks: a snapshot of values and the logic to manipulate them. It has the nice feature that if you distribute a workspace, you can be sure that users of that workspace sees exactly what you intended: it's an exact snapshot.

Let's try this out. In a new `Ride` session, let's create a few functions and variables, like so:
![workspace1](./IMG/ws1.png)

To save the workspace into a workspace file, type

    )save myworkspace
    Cannot perform operation with trace/edit windows open.
    
Ooops, that didn't work. Dyalog can only save a workspace if you close down all edit windows first. So let's do that, and once you've saved your workspace, close down the session and reopen a fresh one.

To get back your workspace, type 

    )load myworkspace
   
![workspace2](./IMG/ws2.png)

We can use the commands `)vars` to see the defined variables, and `)fns` to show the names of any functions contained in the workspace. As we can see, they're all present and correct. If you find yourself using a set of utility functions over and over, they might be usefully combined into a workspace that you can then load up as a unit with a simple `)load` command. As you might expect, the workspace concept is deeply engrained in Dyalog and it comes with rich, sophisticated, thoroughly battle-tested functionality -- all of which is overkill for our needs right now. If you want to explore this further, Legrand's book, [Mastering Dyalog APL](https://www.dyalog.com/uploads/documents/MasteringDyalogAPL.pdf) covers it extensively.

## Using Link

[Link](https://dyalog.github.io/link/3.0/) is a Dyalog library that can be used to set up synchronisation between a directory of text files containing APL source code, and namespaces in your current workspace. As you may recall from the chapter on [namespaces](./namespaces.ipynb), one way to think of them is as a grouping mechanism akin to a module or class. Recent Dyalog versions come with Link bundled, but it's under active development and you can always install the latest version from the above link.

The Link concept is pretty intuitive: a Linked directory becomes a namespace, and all its content loaded. If you make a change to a file using an external tool or editor, that change will be reflected inside Dyalog, and if you make a change to the namespace inside Dyalog, these changes are reflected on disk. This means that you can write APL code in Emacs or VS Code, and have your Dyalog session see any changes you make. And -- of course -- now you can put your code in github with all that this entails in terms of your workflow. 

Let's work through a complete example, step by step, by building a "dict" -- an array that gives the illusion of being indexable by string. Yes, you're not the only one wishing Dyalog had one of those. Start by creating a new directory called `dict` and make a note of the full path:

    stefan-% mkdir -p ~/work/dyalog/dict

Now, using an editor of your choice, you're going to create five functions, each contained in a file with the `.aplf` extension (for APL function). Here's me editing the first one in VS Code, called `create.aplf`:

![dict1](./IMG/dict1.png)

```{note}
One thing you'll notice when editing APL code in VS Code is that as VS Code also uses the option key for certain commands, it becomes a bit more tricky to type APL glyphs. You can always use the backtick as the APL key for combos that VS Code hijacks.
```

Here are the functions themselves, leaving aside for the moment how they work (or indeed if they're particularly good):

In [ ]:
Append←{⎕IO←0
    ⍝ Given a vector of keys and a vector of vals append vals
    ⍺∘{
        k←0⌷⍵ ⋄ v←1⌷⍵
        i←⍺.Keys⍳k
        i=≢⍺.Keys:1⊣⍺.Keys,←k⊣⍺.Vals,←v  ⍝ New key
        ⍺.Vals[i],←v                     ⍝ Append to existing
        0
    }¨↓⍉↑⍵
}

In [ ]:
Create←{
    keys←(1500⌶),⍺
    vals←,⍵
    {⍵⊣⍵.(Vals Keys Default)←vals keys ⍬}⎕NS''
}

In [ ]:
Get←{
    ~⍵ In ⍺:⍺.Default
    ⍺.Vals[⍺.Keys⍳⍵]
}

In [4]:
In←{⍺∊⍵.Keys}

In [ ]:
Set←{⎕IO←0
    ⍝ Upsert ht ⍺ given a vector of keys and a vector of vals
    ⍺∘{
        k←0⌷⍵ ⋄ v←1⌷⍵
        i←⍺.Keys⍳k
        i=≢⍺.Keys:1⊣⍺.Keys,←k⊣⍺.Vals,←v ⍝ New key
        ⍺.Vals[i]←v                     ⍝ Replace existing
        0
    }¨↓⍉↑⍵
}

Now open a new Dyalog session, and link this directory:

```
    ]LINK.Create dict /path/to/dict
```

You should see the following:

    Linked: #.dict → /path/to/dict

All being well, our session should now be aware of the code we just wrote. We can list the functions that Dyalog now thinks the namespace contains by executing the _Name List_ ([⎕NL](https://dyalog.github.io/documentation/21.0/language-reference-guide/system-functions/nl/)) sysfn with a suitable magic number:

```
    dict.⎕NL ¯3
Append  Create  Get  In  Set
```
    
Our functions have clearly been loaded. Let's see if they work, shall we?

![link2](./IMG/link2.png)

## Link limitations

That works really well so far. Any change we make in `Ride` to the `dict` namespace will be reflected on disk, including new functions, or modifications of existing. Bi-directional sync is really what we want, so that if the files change on disk -- for example if you're editing a function with an external editor, that change will be picked up by the interpreter. This works out of the box on Windows, but for Linux and macOS, for Dyalog v21, you will first need to install [.NET 10.0](https://dotnet.microsoft.com/en-us/download/dotnet/10.0), as the file system watcher for Link relies on .NET. If you're an [AIX](https://en.wikipedia.org/wiki/IBM_AIX) user of Dyalog APL, you're out of luck, as .NET isn't available for AIX. 

If you -- as a linux or macOS user -- are thinking "Why the hell should I have to install a completely separate development toolchain I have zero interest in just so that Dyalog APL can see if I changed a file?", well, it's a fair point. Picking up file system change notifications from the OS in a platform independent way is simple in .NET and not simple in any other way that is easily integrated with Dyalog APL. If you think of .NET as Dyalog's core extension mechanism (we've deemed that out of scope for this book), perhaps you can see why Dyalog went down this route. Dyalog's longer term plans have always been to let the interpreter eventually take over Link's job. 

The other thing is that when you're working this way, you'll need to keep each function, namespace, class, or operator in its own separate file, and they all have their separate file extensions, which if you're anything like me is quite a fine granularity, especially since APL functions tend to be pretty short. If Dyalog's gazillion different file extensions rub up wrongly against your core sensibilities, well, that is fair cop, too. 

Finally, and most importantly, Link is NOT an application _deployment_ mechanism, no matter how much it can be dressed up as one. _Never_ distribute a system to end user that relies on Link to load code into a workspace. Bad things will eventually happen. Link is intended as a tool for the development workflow. Don't use Link to simulate the import functionality you wish that Dyalog had.

Another thing is that Links file system notification system sometimes gets overwhelmed by too many rapid changes. If you have a linked directory of many thousands of files and you suddenly do a `git switch` to a different branch, the resulting state in the interpreter may be ... ill-defined. Prefer to unlink before the switch, and then re-link. 

## Dyalogscript

Introduced in Dyalog v18.1, the `dyalogscript` executable allows you to execute a text file containing APL code, top to bottom, with function definitions and all, from the command-line. This means that with this approach you can use Dyalog for 'quick and dirty' scripts where you otherwise would have reached for Python, and that you can easily share code as files, or via a github gist and have other people run it. `dyalogscript` holds a lot of promise -- especially in the AI era -- but we need to be clear about what it is, and what it is not.

It is _not_ designed to be a way to conveniently organise your code into 'modules' held in text files that can easily reference each other. Its syntax rules are different to normal APL code: you're perfectly allowed to use tradfn statements in the middle of a `dyalogscript`, for example. 

For a mental model of what a `dyalogscript` is, think of it as equivalent to a log of something that was typed in the session, fed to the interpreter and evaluated, line by line, as typed. Yes, this means that a file containing only comment lines will take a long time to "run".

That is not to say that `dyalogscript` can't be useful -- it's _very_ useful, for example as a way to give LLMs a [skill](https://github.com/dyalog-labs/agent-dev-container/blob/main/.devcontainer/kit/.claude/skills/dyalog-script/SKILL.md) to evaluate APL snippets. If you wanted a way to write large applications with full control of the dependency graph, `dyalogscript` is not plugging that gap in Dyalog's arsenal. 

By default, `dyalogscript` does not run the normal session initialisation -- probably how you want it if all you're doing is using it as a quick and dirty executor on the command line. If you do want the full session init, make your shebang line look like so:

```sh
#!/usr/local/bin/dyalogscript DYALOG_INITSESSION=1
```

That means that you get the full `⎕SE` etc. 


For example, I created the following file:

```apl
#!/usr/bin/env dyalogscript
⍝ Draw the shortest way through an ASCII maze: '#' is wall, anything else is open.
⍝ Mazes from https://www.dcode.fr/maze-generator

⎕IO←0
'span' 'stpath'⎕CY'dfns'

Graph ← {                                     ⍝ ⍵:open-cell mask → dfns adjacency list
    id ← (⍴⍵)⍴(+\,⍵)×,⍵                       ⍝ number the openings 1…n, walls stay 0
    nb ← ↑,¨(0⍪¯1 0↓id)(0,0 ¯1↓id)((0 1↓id),0)((1 0↓id)⍪0)   ⍝ up, left, right, down
    {¯1+⍵/⍨0≠⍵}¨↓⍉nb[;⍸,⍵]                   ⍝ per opening: the neighbours that exist
}

Ends ← {                                      ⍝ ⍵:maze → coordinates of start and goal
    2=≢m←⍸~⍵∊'#.': m                          ⍝ the two lettered squares if marked…
    c[0,¯1+≢c←⍸~'#'=⍵]                        ⍝ …otherwise first and last opening
}

Solve ← {                                     ⍝ ⍵:maze → maze with the path drawn on
    cells ← ⍸open←~'#'=⍵                      ⍝ node k is the k'th opening, row-major
    (start goal) ← cells⍳Ends ⍵
    '⎕'@(cells[((Graph open)span start)stpath goal])⊢⍵
}

argv ← 2⎕NQ'.' 'GetCommandLineArgs'
file ← ⊃(1↓argv),⊂(⊃⎕NPARTS⊃argv),'maze-small.txt'   ⍝ default: next to the script

⎕ ← Solve↑⊃⎕NGET file 1
```

and can run it from the command line:

![trm](./IMG/term.png)

## VS Code

Microsoft's programmer's editor [VS Code](https://code.visualstudio.com/) is the most widely deployed IDE in the world. Statistically speaking, as a software developer, you probably have it installed on the machine you're reading this on. Dyalog has an unofficial [APL extension for VS Code](https://github.com/dyalog-labs/vscode-apl), and it pulls off an impressive feat: it manages to fit Dyalog APL into the standard VS Code workflow in terms of tracing, file handling etc, without sacrificing the dynamic nature of the session. It has a session pane that maintains a persistent connection to a running Dyalog APL session, and for AI fans, this can be shared with whatever LLM agent which happens to be en vogue (a moving goal post). 

The interesting thing is that all of that was achieved without any changes to the Dyalog APL interpreter itself. Here are a couple of screnshots of this in action:

![vscode1](./IMG/vscode1.png)

I've started the interpreter 'inside' VS Code by clicking the <kbd>Connect</kbd> button in the APL window, and typed the convenience command `/link` to load up the contents of the open folder. Any modifications I make to files will be picked up by the interpreter via Link. 

The extension has full DAP support, too, meaning that you can take advantage of VS Code's normal debugger tools to trace through your Dyalog APL code:

![vscode2](./IMG/vscode2.png)

Many -- maybe even most -- software projects today tend to be heterogeneous: perhaps your backends are written in Go, speaking [SQL](https://en.wikipedia.org/wiki/SQL) to a [postgres](https://www.postgresql.org/) system of record, deployed on [Kubernetes](https://kubernetes.io/), exposing a json-http API, picked up by a javascript [SPA](https://en.wikipedia.org/wiki/Single-page_application). Integrating Dyalog APL components in such architectures fits well, and being able to stay in VS Code for all parts of the application architecture feels more ergonomic perhaps than switching out to a separate toolchain for the Dyalog bits.

The VS Code extension is still young, but if you're working in a mixed environment, it will probably fit the bill. 